# Shared Helpers

This notebook centralizes the reusable setup and data-preparation logic for the project notebooks. Run it directly only if you want to inspect or modify the helper functions; the analysis notebooks load it with `%run ./00_helpers.ipynb`.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 12


In [ ]:
def get_project_root(marker: str = "README.md") -> Path:
    current = Path.cwd().resolve()
    for path in [current, *current.parents]:
        if (path / marker).exists():
            return path
    return current


PROJECT_ROOT = get_project_root()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
DEFAULT_PROCESSED_PATH = PROCESSED_DATA_DIR / "google_playstore_cleaned.csv"


def find_dataset_path(filename: str = "Google-Playstore.csv") -> Path:
    candidates = [
        RAW_DATA_DIR / filename,
        PROJECT_ROOT / filename,
        Path("/kaggle/input/google-playstore-apps") / filename,
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate

    raw_csvs = sorted(RAW_DATA_DIR.glob("*.csv"))
    if raw_csvs:
        return raw_csvs[0]

    raise FileNotFoundError(
        "Dataset not found. Place the raw CSV in data/raw/ or update the path in the notebook."
    )


In [ ]:
COLUMNS_TO_DROP = [
    "Developer Website",
    "App Id",
    "Developer Email",
    "Privacy Policy",
    "Installs",
]

CURRENCY_TO_REGION = {
    "USD": "North America",
    "XXX": "Unknown",
    "CAD": "North America",
    "EUR": "Europe",
    "INR": "Asia",
    "VND": "Asia",
    "GBP": "Europe",
    "BRL": "South America",
    "KRW": "Asia",
    "TRY": "Europe",
    "SGD": "Asia",
    "AUD": "Oceania",
    "ZAR": "Africa",
}

BOOL_COLUMNS = [
    "Free",
    "Ad Supported",
    "In App Purchases",
    "Editors Choice",
]

CATEGORICAL_COLUMNS = [
    "Category",
    "Content Rating",
    "Region",
    "Year",
    "Season",
]

MODEL_FEATURES = [
    "Price",
    "Size",
    "Age",
    "Days Since Update",
    "Rating Confidence",
    "Monetization Score",
    "Free",
    "Ad Supported",
    "In App Purchases",
    "Editors Choice",
    "Category",
    "Content Rating",
    "Region",
    "Year",
    "Season",
]


In [ ]:
def convert_size_to_mb(value):
    if pd.isna(value):
        return np.nan

    text = str(value).strip().replace(",", "")
    if not text or "Varies with" in text:
        return np.nan

    suffix = text[-1]
    factor_map = {"k": 1 / 1024, "M": 1, "G": 1024}

    if suffix in factor_map:
        return float(text[:-1]) * factor_map[suffix]

    return float(text)


def get_season(month):
    if month in [3, 4, 5]:
        return "Spring"
    if month in [6, 7, 8]:
        return "Summer"
    if month in [9, 10, 11]:
        return "Autumn"
    return "Winter"


In [ ]:
def load_raw_playstore_data(path: str | Path | None = None) -> pd.DataFrame:
    dataset_path = Path(path) if path else find_dataset_path()
    print(f"Loading dataset from: {dataset_path}")
    return pd.read_csv(dataset_path)


def clean_playstore_data(raw_df: pd.DataFrame) -> pd.DataFrame:
    df = raw_df.copy()
    existing_drop_columns = [column for column in COLUMNS_TO_DROP if column in df.columns]
    df = df.drop(columns=existing_drop_columns)

    rename_map = {
        "Minimum Installs": "Installs Category",
        "Maximum Installs": "Installs",
    }
    df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

    df = df.dropna().drop_duplicates().reset_index(drop=True)

    if "Currency" in df.columns:
        df["Region"] = df["Currency"].map(CURRENCY_TO_REGION).fillna("Other")
        df = df.drop(columns=["Currency"])

    df["Size"] = df["Size"].apply(convert_size_to_mb)
    df["Size"] = df["Size"].fillna(df["Size"].mean())

    df["Released"] = pd.to_datetime(df["Released"], errors="coerce")
    df["Last Updated"] = pd.to_datetime(df["Last Updated"], errors="coerce")
    scraped_time = df["Scraped Time"].astype(str).str.replace(" UTC", "", regex=False)
    df["Scraped Time"] = pd.to_datetime(scraped_time, errors="coerce")

    df = df.dropna(subset=["Released", "Last Updated", "Scraped Time"]).reset_index(drop=True)

    df["Year"] = df["Released"].dt.year.astype(int)
    df["Age"] = (df["Scraped Time"] - df["Released"]).dt.days / 365.25
    df["Days Since Update"] = (df["Scraped Time"] - df["Last Updated"]).dt.days / 365.25

    max_rating = 5
    rating_counts = df["Rating Count"].clip(lower=1)
    df["Rating Confidence"] = np.where(
        df["Rating Count"] > 0,
        (df["Rating"] * np.log(rating_counts)) / (max_rating * np.log(max_rating)),
        0.0,
    )

    df["Season"] = df["Released"].dt.month.apply(get_season)

    ad_revenue_per_install = 0.05
    iap_revenue_per_install = 0.10
    max_revenue = 600000000
    revenue = np.where(
        df["Free"],
        df["Installs"] * (
            (df["Ad Supported"] * ad_revenue_per_install)
            + (df["In App Purchases"] * iap_revenue_per_install)
        ),
        df["Installs"] * df["Price"],
    )
    df["Monetization Score"] = revenue / np.log(max_revenue)

    for column in BOOL_COLUMNS:
        if column in df.columns:
            df[column] = df[column].astype(int)

    return df


def load_or_prepare_data(processed_path: str | Path | None = None, raw_path: str | Path | None = None) -> pd.DataFrame:
    processed = Path(processed_path) if processed_path else DEFAULT_PROCESSED_PATH
    if processed.exists():
        print(f"Loading prepared dataset from: {processed}")
        return pd.read_csv(processed, parse_dates=["Released", "Last Updated", "Scraped Time"])

    raw_df = load_raw_playstore_data(raw_path)
    return clean_playstore_data(raw_df)


In [ ]:
def save_prepared_data(df: pd.DataFrame, output_path: str | Path | None = None) -> Path:
    output = Path(output_path) if output_path else DEFAULT_PROCESSED_PATH
    output.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output, index=False)
    print(f"Saved prepared dataset to: {output}")
    return output


def display_shape_change(raw_df: pd.DataFrame, clean_df: pd.DataFrame) -> pd.DataFrame:
    summary = pd.DataFrame(
        {
            "rows": [len(raw_df), len(clean_df)],
            "columns": [raw_df.shape[1], clean_df.shape[1]],
        },
        index=["raw", "prepared"],
    )
    summary["rows_removed"] = [0, len(raw_df) - len(clean_df)]
    return summary


def sample_for_scatter(df: pd.DataFrame, n: int = 5000, random_state: int = 42) -> pd.DataFrame:
    sampled = df.sample(n=min(n, len(df)), random_state=random_state).copy()
    sampled["log_installs"] = np.log1p(sampled["Installs"])
    return sampled
